# Tensor Mode Solver: Spin-Torsion Bounce

Compute the tensor perturbation spectrum through a radiation-dominated
spin-torsion bounce.

**Background:** $a(t) = a_b(1 + 4\alpha^2 t^2)^{1/4}$ with $\alpha^2 = 8\pi G \rho_{\rm crit}/3$

**Mode equation:** $v_k'' + (k^2 - a''/a)\,v_k = 0$

**Goal:** Extract Bogoliubov coefficients $\beta_k$ and power spectrum $P_T(k)$

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.special import gamma
import matplotlib.pyplot as plt

# Natural units: M_Pl = 1, so alpha^2 = 8*pi*rho_crit/3
# with rho_crit ~ 0.21 M_Pl^4
rho_crit = 0.21  # in M_Pl^4
alpha2 = 8 * np.pi * rho_crit / 3  # ~ 1.76 M_Pl^2
alpha = np.sqrt(alpha2)

# Set a_b = 1 (in Planck units; physical a_b ~ l_Pl)
a_b = 1.0

# Characteristic bounce wavenumber
k_b = a_b * np.sqrt(2) * alpha

print(f"alpha = {alpha:.4f} M_Pl")
print(f"alpha^2 = {alpha2:.4f} M_Pl^2")
print(f"k_b = {k_b:.4f} a_b M_Pl")
print(f"k_b/a_b = {k_b/a_b:.4f} M_Pl  (physical momentum at bounce)")

## 1. Background Solution

In [ ]:
def a_cosmic(t):
    """Scale factor as function of cosmic time."""
    return a_b * (1 + 4 * alpha2 * t**2)**0.25

def H_cosmic(t):
    """Hubble parameter as function of cosmic time."""
    return 2 * alpha2 * t / (1 + 4 * alpha2 * t**2)

def Hdot_cosmic(t):
    """dH/dt."""
    u = 4 * alpha2 * t**2
    return 2 * alpha2 * (1 - u) / (1 + u)**2

def app_over_a_cosmic(t):
    """a''/a in conformal time, expressed as function of cosmic time.
    a''/a = a^2(Hdot + 2H^2)"""
    a = a_cosmic(t)
    return 2 * alpha2 * a_b**2 / (1 + 4 * alpha2 * t**2)**1.5

# Plot background
t_arr = np.linspace(-3/alpha, 3/alpha, 1000)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(t_arr * alpha, a_cosmic(t_arr)/a_b)
axes[0,0].set_xlabel(r'$\alpha t$')
axes[0,0].set_ylabel(r'$a/a_b$')
axes[0,0].set_title('Scale Factor')

axes[0,1].plot(t_arr * alpha, H_cosmic(t_arr)/alpha)
axes[0,1].set_xlabel(r'$\alpha t$')
axes[0,1].set_ylabel(r'$H/\alpha$')
axes[0,1].set_title('Hubble Parameter')

axes[1,0].plot(t_arr * alpha, Hdot_cosmic(t_arr)/alpha2)
axes[1,0].set_xlabel(r'$\alpha t$')
axes[1,0].set_ylabel(r'$\dot{H}/\alpha^2$')
axes[1,0].set_title(r'$\dot{H}$')

axes[1,1].plot(t_arr * alpha, app_over_a_cosmic(t_arr)/(alpha2 * a_b**2))
axes[1,1].set_xlabel(r'$\alpha t$')
axes[1,1].set_ylabel(r"$a''/a \;/\; (\alpha^2 a_b^2)$")
axes[1,1].set_title('Effective Potential')

plt.tight_layout()
plt.savefig('background_solution.png', dpi=150)
plt.show()

## 2. Conformal Time Relation

Numerically compute $\eta(t)$ and then express everything in conformal time.

In [ ]:
def compute_conformal_time(t_array):
    """Compute conformal time eta for each cosmic time t.
    deta = dt / a(t)"""
    eta = np.zeros_like(t_array)
    for i in range(1, len(t_array)):
        dt = t_array[i] - t_array[i-1]
        # Trapezoidal rule
        eta[i] = eta[i-1] + 0.5 * dt * (1/a_cosmic(t_array[i-1]) + 1/a_cosmic(t_array[i]))
    return eta

# Dense time grid spanning -20/alpha to +20/alpha
N_t = 100000
t_grid = np.linspace(-20/alpha, 20/alpha, N_t)
eta_grid = compute_conformal_time(t_grid)

# Compute a''/a on the conformal time grid
U_grid = np.array([app_over_a_cosmic(t) for t in t_grid])

# Check: U should be localized near eta=0
plt.figure(figsize=(10, 5))
plt.plot(eta_grid * alpha * a_b, U_grid / (alpha2 * a_b**2))
plt.xlabel(r'$\alpha a_b \eta$')
plt.ylabel(r"$a''/a \;/\; (\alpha^2 a_b^2)$")
plt.title('Effective Potential in Conformal Time')
plt.xlim(-5, 5)
plt.savefig('effective_potential.png', dpi=150)
plt.show()

print(f"Conformal time range: [{eta_grid[0]:.4f}, {eta_grid[-1]:.4f}] / (a_b/alpha)")

## 3. Solve Tensor Mode Equation

$v_k'' + (k^2 - U(\eta))\,v_k = 0$

Initial condition: $v_k = \frac{1}{\sqrt{2k}} e^{-ik\eta}$ in the far past.

In [ ]:
from scipy.interpolate import interp1d

# Interpolate the potential as function of conformal time
U_interp = interp1d(eta_grid, U_grid, kind='cubic', fill_value=0.0,
                     bounds_error=False)

def solve_tensor_mode(k_val, eta_span, eta_eval, U_func):
    """Solve v'' + (k^2 - U(eta)) v = 0.
    
    Initial condition: v = e^{-ik*eta}/sqrt(2k) at eta_start
    (positive frequency vacuum).
    
    Returns (eta_eval, v_real, v_imag).
    System: y = [Re(v), Im(v), Re(v'), Im(v')]
    """
    omega2 = k_val**2
    eta0 = eta_span[0]
    
    # Initial conditions: v = e^{-ik*eta0}/sqrt(2k)
    norm = 1.0 / np.sqrt(2 * k_val)
    v0_re = norm * np.cos(-k_val * eta0)
    v0_im = norm * np.sin(-k_val * eta0)
    vp0_re = norm * k_val * np.sin(-k_val * eta0)   # d/deta of cos(-k*eta) * norm
    vp0_im = -norm * k_val * np.cos(-k_val * eta0)  # d/deta of sin(-k*eta) * norm
    
    def rhs(eta, y):
        v_re, v_im, vp_re, vp_im = y
        U_val = U_func(eta)
        return [
            vp_re,
            vp_im,
            -(omega2 - U_val) * v_re,
            -(omega2 - U_val) * v_im
        ]
    
    sol = solve_ivp(rhs, eta_span, [v0_re, v0_im, vp0_re, vp0_im],
                    t_eval=eta_eval, rtol=1e-10, atol=1e-12,
                    method='DOP853')
    
    return sol.t, sol.y[0] + 1j * sol.y[1], sol.y[2] + 1j * sol.y[3]

print("Solver defined.")

In [ ]:
def extract_bogoliubov(k_val, v_out, vp_out, eta_out):
    """Extract Bogoliubov coefficients from the output mode function.
    
    In the far future: v = alpha_k * e^{-ik*eta}/sqrt(2k) + beta_k * e^{+ik*eta}/sqrt(2k)
    v' = -ik*alpha_k * e^{-ik*eta}/sqrt(2k) + ik*beta_k * e^{+ik*eta}/sqrt(2k)
    
    Solving for alpha and beta at a given eta:
    alpha_k = sqrt(2k)/2 * (v + v'/(ik)) * e^{ik*eta}
            = sqrt(2k)/2 * (v - i*v'/k) * e^{ik*eta}
    beta_k  = sqrt(2k)/2 * (v - v'/(ik)) * e^{-ik*eta}
            = sqrt(2k)/2 * (v + i*v'/k) * e^{-ik*eta}
    """
    norm = np.sqrt(2 * k_val) / 2
    
    alpha_arr = norm * (v_out - 1j * vp_out / k_val) * np.exp(1j * k_val * eta_out)
    beta_arr = norm * (v_out + 1j * vp_out / k_val) * np.exp(-1j * k_val * eta_out)
    
    return alpha_arr, beta_arr

print("Bogoliubov extractor defined.")

## 4. Compute β_k for a Range of k Values

In [ ]:
# Solve for a range of k values
k_values = np.logspace(-2, 1.5, 60) * k_b  # from 0.01 k_b to ~30 k_b

# Use the conformal time grid
# Start well before bounce, end well after
eta_start = eta_grid[0]
eta_end = eta_grid[-1]
n_eval = 5000
eta_eval = np.linspace(eta_start, eta_end, n_eval)

beta_sq = []
alpha_vals = []

print(f"Solving for {len(k_values)} modes...")
print(f"eta range: [{eta_start:.4f}, {eta_end:.4f}]")
print(f"k_b = {k_b:.4f}")

for i, k in enumerate(k_values):
    try:
        eta_sol, v_sol, vp_sol = solve_tensor_mode(
            k, [eta_start, eta_end], eta_eval, U_interp)
        
        # Extract Bogoliubov at the last ~10% of eta range (far from bounce)
        n_tail = len(eta_sol) // 10
        alpha_k, beta_k = extract_bogoliubov(
            k, v_sol[-n_tail:], vp_sol[-n_tail:], eta_sol[-n_tail:])
        
        # Should be approximately constant in the tail
        beta_sq_val = np.mean(np.abs(beta_k)**2)
        alpha_val = np.mean(np.abs(alpha_k)**2)
        
        # Check normalization: |alpha|^2 - |beta|^2 = 1
        norm_check = alpha_val - beta_sq_val
        
        beta_sq.append(beta_sq_val)
        alpha_vals.append(alpha_val)
        
        if i % 10 == 0:
            print(f"  k/k_b = {k/k_b:.3f}, |beta|^2 = {beta_sq_val:.4e}, "
                  f"|alpha|^2 - |beta|^2 = {norm_check:.6f}")
    except Exception as e:
        print(f"  k/k_b = {k/k_b:.3f}: FAILED ({e})")
        beta_sq.append(np.nan)
        alpha_vals.append(np.nan)

beta_sq = np.array(beta_sq)
alpha_vals = np.array(alpha_vals)

print(f"\nDone. Max |beta|^2 = {np.nanmax(beta_sq):.2f}")
print(f"Min |beta|^2 (k >> k_b) = {np.nanmin(beta_sq[k_values > 3*k_b]):.2e}")

In [ ]:
# Plot |beta_k|^2 vs k/k_b
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mask = ~np.isnan(beta_sq)

# Log-log
axes[0].loglog(k_values[mask]/k_b, beta_sq[mask], 'b.-')
axes[0].set_xlabel(r'$k / k_b$')
axes[0].set_ylabel(r'$|\beta_k|^2$')
axes[0].set_title('Bogoliubov coefficient (graviton production)')
axes[0].axhline(1, color='gray', ls='--', alpha=0.5, label=r'$|\beta|^2 = 1$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Linear
axes[1].semilogy(k_values[mask]/k_b, beta_sq[mask], 'b.-')
axes[1].set_xlabel(r'$k / k_b$')
axes[1].set_ylabel(r'$|\beta_k|^2$')
axes[1].set_title('Bogoliubov coefficient (linear k axis)')
axes[1].axhline(1, color='gray', ls='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bogoliubov_coefficients.png', dpi=150)
plt.show()

# Report key values
idx_low = k_values < 0.1 * k_b
if np.any(idx_low & mask):
    print(f"\n|beta|^2 at k << k_b: {np.mean(beta_sq[idx_low & mask]):.2f}")
print(f"|beta|^2 at k = k_b: {beta_sq[mask][np.argmin(np.abs(k_values[mask] - k_b))]:.2f}")

## 5. Tensor Power Spectrum

$$P_T(k) = \frac{4k^2}{\pi^2 M_{\rm Pl}^2} \cdot (1 + 2|\beta_k|^2) \cdot \frac{1}{a^2}$$

The factor $1/a^2$ is the overall dilution. The physically meaningful quantity
for comparison is the **dimensionless spectrum** shape:

$$\mathcal{P}_T(k) \propto k^2 \cdot (1 + 2|\beta_k|^2)$$

In [ ]:
# Dimensionless tensor power spectrum (shape only, in natural units M_Pl=1)
P_T = k_values**2 * (1 + 2 * beta_sq)

# Normalize to peak
P_T_norm = P_T / np.nanmax(P_T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-log
axes[0].loglog(k_values[mask]/k_b, P_T[mask], 'r.-')
axes[0].set_xlabel(r'$k / k_b$')
axes[0].set_ylabel(r'$P_T(k)$ [arb. units]')
axes[0].set_title('Tensor Power Spectrum')

# Overlay k^2 scaling for reference
k_ref = k_values[mask]
k2_line = (k_ref/k_b)**2 * P_T[mask][np.argmin(np.abs(k_values[mask] - 0.03*k_b))]
axes[0].loglog(k_ref/k_b, k2_line, 'k--', alpha=0.5, label=r'$\propto k^2$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Normalized
axes[1].semilogx(k_values[mask]/k_b, P_T_norm[mask], 'r.-')
axes[1].set_xlabel(r'$k / k_b$')
axes[1].set_ylabel(r'$P_T(k) / P_T^{\rm max}$')
axes[1].set_title('Normalized Tensor Power Spectrum')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tensor_power_spectrum.png', dpi=150)
plt.show()

# Find peak
idx_peak = np.nanargmax(P_T)
print(f"\nPeak at k/k_b = {k_values[idx_peak]/k_b:.3f}")
print(f"Peak P_T value = {P_T[idx_peak]:.4e}")

## 6. Spectral Index

$$n_T(k) = \frac{d \ln P_T}{d \ln k}$$

In [ ]:
# Compute spectral index by numerical differentiation
ln_k = np.log(k_values[mask])
ln_PT = np.log(P_T[mask])

# Central differences
n_T = np.gradient(ln_PT, ln_k)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(k_values[mask]/k_b, n_T, 'g.-', markersize=4)
ax.axhline(2, color='red', ls='--', alpha=0.7, label=r'$n_T = 2$')
ax.axhline(0, color='black', ls='-', alpha=0.3)
ax.set_xlabel(r'$k / k_b$')
ax.set_ylabel(r'$n_T = d\ln P_T / d\ln k$')
ax.set_title('Tensor Spectral Index')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 4)
plt.tight_layout()
plt.savefig('spectral_index.png', dpi=150)
plt.show()

# Report values
idx_low_k = (k_values[mask] < 0.1 * k_b) & (k_values[mask] > 0.02 * k_b)
if np.any(idx_low_k):
    print(f"n_T at k << k_b: {np.mean(n_T[idx_low_k]):.3f}  (expect ~2)")

idx_near_kb = (k_values[mask] > 0.5*k_b) & (k_values[mask] < 2*k_b)
if np.any(idx_near_kb):
    print(f"n_T near k_b: {np.mean(n_T[idx_near_kb]):.3f}")

idx_high_k = k_values[mask] > 3*k_b
if np.any(idx_high_k):
    print(f"n_T at k >> k_b: {np.mean(n_T[idx_high_k]):.3f}  (expect negative)")

## 7. Analytic Cross-Check: WKB Tunneling Estimate

For $k = 0$, the eikonal integral gives:

$$\sigma = \frac{\sqrt{2}}{2} \int_{-\infty}^{\infty} du\,(1+u^2)^{-3/4} = \frac{\sqrt{2}}{2} \cdot \frac{\sqrt{\pi}\,\Gamma(1/4)}{\Gamma(3/4)}$$

In [ ]:
# Analytic estimate for |beta_0|^2
from scipy.special import gamma as gamma_func

integral_val = np.sqrt(np.pi) * gamma_func(0.25) / gamma_func(0.75)
sigma = np.sqrt(2) / 2 * integral_val

beta0_sq_analytic = np.sinh(sigma)**2

print(f"Eikonal integral: {integral_val:.4f}")
print(f"sigma = {sigma:.4f}")
print(f"|beta_0|^2 (WKB) = sinh^2(sigma) = {beta0_sq_analytic:.2f}")

# Compare with numerical result at lowest k
if np.any(mask):
    idx_lowest = np.argmin(k_values[mask])
    print(f"|beta|^2 (numerical, k/k_b = {k_values[mask][idx_lowest]/k_b:.3f}) = "
          f"{beta_sq[mask][idx_lowest]:.2f}")
    print(f"\nRatio numerical/analytic = {beta_sq[mask][idx_lowest]/beta0_sq_analytic:.3f}")

## 8. Observability: Frequency Mapping

Map comoving $k$ to observed frequency today using:
$$f = \frac{k}{2\pi a_0}$$

with $a_b/a_0 \approx 9.3 \times 10^{-33}$.

In [ ]:
# Redshift from bounce to today
# T_bounce ~ 0.68 M_Pl ~ 8.3e27 eV
# T_0 = 2.35e-4 eV
# g_s ratio: (3.91/106.75)^{1/3} = 0.332
a_b_over_a0 = 0.332 * 2.35e-4 / (0.68 * 1.22e28)  # eV/eV
print(f"a_b/a_0 = {a_b_over_a0:.2e}")

# k_b in Hz today
# k_b/a_b = sqrt(2)*alpha ~ 1.88 M_Pl
# f_b = k_b/(2*pi*a_0) = (k_b/a_b) * (a_b/a_0) / (2*pi)
# In Hz: M_Pl = 1.22e19 GeV = 1.22e28 eV, and 1 eV -> 2.42e14 Hz
k_b_physical = np.sqrt(2) * alpha  # in M_Pl units
f_b_Hz = k_b_physical * 1.22e28 * a_b_over_a0 * 2.42e14 / (2 * np.pi)
print(f"f_b = {f_b_Hz:.2e} Hz = {f_b_Hz/1e9:.1f} GHz")

# Detector bands
detectors = {
    'CMB (Hubble scale)': 1e-18,  # Hz
    'PTA (NANOGrav)': 1e-8,
    'LISA': 1e-2,
    'LIGO/Virgo': 100,
    'Einstein Telescope': 3,
}

print(f"\n{'Detector':<25} {'f [Hz]':>12} {'k/k_b':>12} {'P_T/P_T(k_b)':>15}")
print("-" * 70)
for name, f_det in detectors.items():
    k_ratio = f_det / f_b_Hz
    # P_T ~ k^2 * |beta|^2 ~ k^2 * const for k << k_b
    # P_T/P_T_peak ~ (k/k_b)^2 for k << k_b
    suppression = k_ratio**2
    print(f"{name:<25} {f_det:>12.2e} {k_ratio:>12.2e} {suppression:>15.2e}")

## 9. GW Energy Density Spectrum $\Omega_{\rm GW}(f)$

In [ ]:
# The GW energy density per log frequency interval:
# Omega_GW(k) = (1/12) * (k/(aH))^2 * P_T(k)
# 
# At the peak (k = k_b), the amplitude depends on the 
# overall normalization. For the bounce:
# P_T(k_b) ~ (k_b^2 / M_Pl^2) * |beta_{k_b}|^2 / a^2
#
# The energy density today from bounce-produced gravitons:
# Omega_GW(k_b) ~ (1/12)(k_b/(a_0 H_0))^2 * P_T(k_b)

# In Planck units: H_0 ~ 1.5e-42 GeV ~ 1.2e-61 M_Pl
H_0_Mpl = 1.2e-61  # M_Pl

# P_T at peak (in proper units):
# P_T(k_b) = (4 k_b^2)/(pi^2 M_Pl^2 a_0^2) * (1 + 2*|beta_{k_b}|^2)
# Using k_b = 1.88 * a_b * M_Pl, and |beta_{k_b}|^2 ~ 1:
# P_T(k_b) ~ (4 * (1.88)^2 * a_b^2)/(pi^2 * a_0^2) * 3
#           ~ 4.3 * (a_b/a_0)^2

P_T_peak = 4 * (1.88)**2 / np.pi**2 * a_b_over_a0**2 * 3
print(f"P_T at peak ~ {P_T_peak:.2e}")

# Omega_GW at peak:
# Omega_GW ~ (1/12) * (k_b/(a_0*H_0))^2 * P_T(k_b)
k_b_over_a0H0 = 1.88 * a_b_over_a0 * 1 / H_0_Mpl  # k_b = 1.88 a_b M_Pl
Omega_peak = (1/12) * k_b_over_a0H0**2 * P_T_peak
print(f"k_b/(a_0 H_0) ~ {k_b_over_a0H0:.2e}")
print(f"Omega_GW at peak ~ {Omega_peak:.2e}")

# This is the fractional energy density in GWs at the peak.
# Compare with BBN bound: Omega_GW * h^2 < 1.12e-6 (integrated)
# and with detector sensitivities.

# At lower frequencies (f << f_b):
# Omega_GW(f) ~ Omega_peak * (f/f_b)^2 * (f/f_b)^2 = Omega_peak * (f/f_b)^4
# Wait: Omega_GW ~ k^2 * P_T ~ k^2 * k^2 * |beta|^2 ~ k^4 for k << k_b
# (since |beta|^2 ~ const for k << k_b)

print(f"\nSpectral shape: Omega_GW(f) ~ f^4 for f << f_b")
print(f"This is an extremely steep blue spectrum.")

print(f"\n{'Detector':<25} {'Omega_GW':>15}")
print("-" * 45)
for name, f_det in detectors.items():
    ratio = f_det / f_b_Hz
    omega_det = Omega_peak * ratio**4 if ratio < 1 else Omega_peak * np.exp(-4*(ratio-1))
    print(f"{name:<25} {omega_det:>15.2e}")
    
print(f"\nBBN bound: Omega_GW*h^2 < 1.12e-6")
print(f"Peak Omega_GW*h^2 ~ {Omega_peak * 0.674**2:.2e}")

## 10. Summary

In [ ]:
print("="*60)
print("TENSOR PERTURBATION SPECTRUM: SUMMARY")
print("="*60)
print()
print(f"Background: a(t) = a_b(1 + 4*alpha^2*t^2)^(1/4)")
print(f"  alpha = {alpha:.4f} M_Pl")
print(f"  rho_crit = {rho_crit} M_Pl^4")
print()
print(f"Characteristic scale:")
print(f"  k_b = {k_b:.4f} a_b M_Pl")
print(f"  f_b = {f_b_Hz:.1e} Hz ({f_b_Hz/1e9:.1f} GHz)")
print()
print(f"Bogoliubov coefficient:")
print(f"  |beta_0|^2 (WKB) = {beta0_sq_analytic:.1f}")
if np.any(mask):
    print(f"  |beta|^2 at lowest k (numerical) = {beta_sq[mask][0]:.1f}")
print()
print(f"Spectrum shape:")
print(f"  P_T(k) ~ k^2 * |beta_k|^2")
print(f"  For k << k_b:  P_T ~ k^2  (n_T = 2, blue)")
print(f"  For k >> k_b:  P_T -> 0 exponentially")
print(f"  Peak near k ~ k_b")
print()
print(f"Spectral index:")
print(f"  n_T = 2 for k << k_b (BLUE tilt)")
print(f"  (Inflation predicts n_T = -r/8 < 0, RED tilt)")
print()
print(f"Observability:")
print(f"  Peak frequency: ~{f_b_Hz/1e9:.0f} GHz")
print(f"  CMB suppression: (f_CMB/f_b)^2 ~ 10^-56")
print(f"  LIGO suppression: (f_LIGO/f_b)^2 ~ 10^-16")
print(f"  PTA suppression: (f_PTA/f_b)^2 ~ 10^-36")
print()
print(f"Verdict: BLUE-TILTED but peak at ~GHz — unobservable")
print(f"  at all current and planned detector frequencies.")